# ICT-15f -- Pont #1-bis : la famille decouplee tranche le negatif de sigma

Le **Pont #1** (« la stabilite locale $\sigma$ cause la recuperabilite ») a ete **falsifie** sur la fronce de Thom (PR #8944, notebook ICT-15e / ICT-8 §8) : la courbure locale $\sigma = V''(x^*)$ n'a aucun pouvoir predictif propre (correlation partielle $\approx 0$) une fois la **largeur de bassin** controlee, et c'est la largeur (position du col) qui gouverne la portee de recuperation ($\rho \approx 0{,}99$).

**Mais** sur la fronce, $\sigma$ et la largeur sont **couplees par construction** ($\rho_{\sigma,\text{width}} \approx 0{,}93$). Le verdict de falsification est donc acquis *sur ce substrat* ; sa **generalite reste indecidable** tant qu'on n'a pas un substrat ou les deux quantites varient *independamment*.

Ce notebook construit cette famille decouplee, rejoue le protocole avec une rigueur statistique accrue (**correlation partielle a 2 covariables** : largeur ET barriere), et tranche. Le module reutilisable est [`ict.basin_family`](../ict/basin_family.py).

> **Statut épistémique** — **Sans verdict à ce jour** : aucune ligne de la [matrice de dissociations](../../../../docs/ict/dissociations-matrix.md) ne concerne ce notebook ; son statut épistémique sera porté par la matrice le cas échéant.

## 1. Le substrat : double-puits symetrique $V(x) = a x^4 - b x^2$

Contrairement a la fronce ($V = x^4/4 + ax^2/2 + bx$, deux parametres de controle mais $\sigma$ et largeur lies), le double-puits symetrique separe algebriquement les trois quantites geometriques :

| Quantite | Formule | Depend de |
|----------|---------|-----------|
| Courbure au minimum $\sigma = V''(x^*)$ | $4b$ | $b$ **seul** |
| Demi-largeur vers le col $w$ | $\sqrt{b/(2a)}$ | $a$ et $b$ |
| Hauteur de barriere $\Delta V$ | $b^2/(4a)$ | $a$ et $b$ |

**L'astuce du decouplage** : on echantillonne $\sigma$ et $w$ sur deux grilles *separees*, et chaque couple cible $(\sigma, w)$ se realise par $b = \sigma/4$ puis $a = b/(2w^2)$. Sur le produit cartesien, $\operatorname{corr}(\sigma, w) \approx 0$ par construction.

**Le piege** : la barriere $\Delta V = \sigma \cdot w^2 / 8$ *co-varie* avec le produit des deux cadrans. Controler la largeur seule (comme le Pont #1) laisse donc fuir la barriere. Le verdict decisif controlera **la largeur ET la barriere**.

In [1]:
#r "nuget: ScottPlot, 5.0.55"

using System;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;
using Microsoft.DotNet.Interactive.Formatting;

CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentUICulture = CultureInfo.InvariantCulture;

public class Graphique
{
    public string Svg { get; }
    public Graphique(string svg) { Svg = svg; }
}
Formatter.Register<Graphique>((g, writer) => writer.Write(g.Svg), "image/svg+xml");

public static class Bassins
{
    public static (double a, double b) Realiser(double sigma, double width)
    {
        if (sigma <= 0 || width <= 0) throw new ArgumentOutOfRangeException();
        double b = sigma / 4.0;
        return (b / (2.0 * width * width), b);
    }
    public static double Potentiel(double x, double a, double b) => a * Math.Pow(x, 4) - b * x * x;
    public static double Force(double x, double a, double b) => 2.0 * b * x - 4.0 * a * x * x * x;
}

// Visualisation du potentiel pour quelques couples (sigma, width)
var couples = new (double sigma, double width)[] { (4.0, 0.6), (4.0, 1.5), (10.0, 1.0) };
double[] abscisses = Enumerable.Range(0, 400).Select(i => -2.0 + 4.0 * i / 399.0).ToArray();
foreach (var couple in couples)
{
    var (a, b) = Bassins.Realiser(couple.sigma, couple.width);
    double[] ordonnees = abscisses.Select(x => Bassins.Potentiel(x, a, b)).ToArray();
    var plot = new ScottPlot.Plot();
    plot.Add.Scatter(abscisses, ordonnees);
    plot.Add.HorizontalLine(0);
    plot.Axes.SetLimits(-2, 2, -3, 3);
    plot.Title($"sigma={couple.sigma:F1}, w={couple.width:F1} (Delta V={couple.sigma * couple.width * couple.width / 8:F2})");
    plot.YLabel("V(x)");
    display(new Graphique(plot.GetSvgXml(367, 320)));
}

Installing Packages ScottPlot

Svg


Svg


Svg


Les trois panneaux fixent $\sigma$ puis font varier $w$, et vice-versa : la raideur du minimum (courbure) et la distance au col se reglent *separement*. C'est ce degre de liberte supplementaire qui manquait a la fronce.

## 2. Le decouplage constructif, mesure

Verifions sur la grille par defaut que $\operatorname{corr}(\sigma, w) \approx 0$ — c'est le **pre-requis** sans lequel le verdict de generalite n'a pas de sens.

In [2]:
// La famille decouplee : produit cartesien de deux grilles (sigma, width).
// On mesure les 4 quantites par minimum stable droit (recuperation stochastique).
public class Alea
{
    private ulong etat;
    private bool disponible;
    private double reserve;
    public Alea(int seed) { etat = unchecked((ulong)(uint)seed + 0x9E3779B97F4A7C15UL); }
    public ulong Entier()
    {
        unchecked
        {
            etat ^= etat >> 12;
            etat ^= etat << 25;
            etat ^= etat >> 27;
            return etat * 2685821657736338717UL;
        }
    }
    public double Uniforme() => ((Entier() >> 11) + 0.5) * (1.0 / 9007199254740992.0);
    public double Normale()
    {
        if (disponible) { disponible = false; return reserve; }
        double rayon = Math.Sqrt(-2.0 * Math.Log(Uniforme()));
        double angle = 2.0 * Math.PI * Uniforme();
        reserve = rayon * Math.Sin(angle);
        disponible = true;
        return rayon * Math.Cos(angle);
    }
}

public static class Mesures
{
    public static double Recuperation(double xstar, double a, double b, Alea rng, double noise = 0.35, int nTrials = 60, int T = 3000, double dt = 0.02, double perturbFrac = 0.75)
    {
        double sdt = Math.Sqrt(2.0 * noise * dt);
        int retour = 0;
        for (int essai = 0; essai < nTrials; essai++)
        {
            double x = xstar * perturbFrac;
            for (int t = 0; t < T; t++)
                x = x + Bassins.Force(x, a, b) * dt + sdt * rng.Normale();
            if (x > 0.0) retour++;
        }
        return (double)retour / nTrials;
    }

    public static double[] Rang(double[] xs)
    {
        int[] ordre = Enumerable.Range(0, xs.Length).OrderBy(i => xs[i]).ToArray();
        double[] rangs = new double[xs.Length];
        int i = 0;
        while (i < ordre.Length)
        {
            int j = i;
            while (j + 1 < ordre.Length && xs[ordre[j + 1]] == xs[ordre[i]]) j++;
            double moyen = 0.5 * (i + j) + 1.0;
            for (int k = i; k <= j; k++) rangs[ordre[k]] = moyen;
            i = j + 1;
        }
        return rangs;
    }

    public static double Pearson(double[] xs, double[] ys)
    {
        if (xs.Length < 2) return 0.0;
        double mx = xs.Average(), my = ys.Average();
        double xx = 0, yy = 0, xy = 0;
        for (int i = 0; i < xs.Length; i++)
        {
            double x = xs[i] - mx, y = ys[i] - my;
            xx += x * x; yy += y * y; xy += x * y;
        }
        if (Math.Sqrt(xx / xs.Length) < 1e-12 || Math.Sqrt(yy / ys.Length) < 1e-12) return 0.0;
        return xy / Math.Sqrt(xx * yy);
    }

    public static Dictionary<string, double[]> Famille(double[] sigmas, double[] largeurs, double noise, int nTrials, int T, double dt, double perturbFrac, int seed)
    {
        var rng = new Alea(seed);
        var s = new List<double>(); var w = new List<double>();
        var bar = new List<double>(); var rec = new List<double>();
        foreach (double sigma in sigmas)
        foreach (double largeur in largeurs)
        {
            var (a, b) = Bassins.Realiser(sigma, largeur);
            double xstar = Math.Sqrt(b / (2.0 * a));
            s.Add(4.0 * b); w.Add(xstar); bar.Add(b * b / (4.0 * a));
            rec.Add(Recuperation(xstar, a, b, rng, noise, nTrials, T, dt, perturbFrac));
        }
        return new Dictionary<string, double[]> { ["sigma"] = s.ToArray(), ["width"] = w.ToArray(), ["barrier"] = bar.ToArray(), ["recovery"] = rec.ToArray() };
    }
}

var fam = Mesures.Famille(new[] { 2.0, 3.0, 4.0, 6.0, 9.0, 13.0 }, new[] { 0.4, 0.6, 0.9, 1.3, 1.9 }, 0.35, 60, 3000, 0.02, 0.75, 0);
double rhoSW = Mesures.Pearson(Mesures.Rang(fam["sigma"]), Mesures.Rang(fam["width"]));
double rhoSB = Mesures.Pearson(Mesures.Rang(fam["sigma"]), Mesures.Rang(fam["barrier"]));
double moyenneRec = fam["recovery"].Average();
double ecartRec = Math.Sqrt(fam["recovery"].Select(x => (x - moyenneRec) * (x - moyenneRec)).Average());
Console.WriteLine($"n echantillons : {fam["sigma"].Length}");
Console.WriteLine($"corr(sigma, width)  = {rhoSW:+0.000;-0.000;+0.000}  (decouplage : |.| < 0.2)");
Console.WriteLine($"corr(sigma, barrier)= {rhoSB:+0.000;-0.000;+0.000}  (fuite de barriere)");
Console.WriteLine($"recovery : min={fam["recovery"].Min():F3} max={fam["recovery"].Max():F3} std={ecartRec:F3}  (variance reelle)");

n echantillons : 30


corr(sigma, width)  = +0.000  (decouplage : |.| < 0.2)


corr(sigma, barrier)= +0.463  (fuite de barriere)


recovery : min=0.383 max=1.000 std=0.186  (variance reelle)


**Lecture de la sortie.** Sur les 30 echantillons de la famille : la correlation $\rho(\sigma, w) = +0{,}000$ confirme le decouplage constructif (le critere exige $|\rho| < 0{,}2$) -- par construction du produit cartesien des grilles, courbure et largeur varient independamment, et la mesure le verifie. Mais la deuxieme ligne revele la fuite : $\rho(\sigma, \text{barrier}) = +0{,}463$. La barrier co-varie avec la courbure *malgre* le decouplage $\sigma/w$ -- geometriquement inevitable (raideur de puits et hauteur du col sont lies par la forme $ax^4 - bx^2$). Cette fuite est exactement ce qui interdit de conclure sur la correlation brute : sans purge de la barriere, un effet attribue a $\sigma$ pourrait etre le sien. La troisieme ligne etablit l'enjeu : la recuperation s'etale de $0{,}400$ a $1{,}000$ (ecart-type $0{,}181$) -- il y a de la variance reelle a expliquer, la question de savoir *qui* la porte est ouverte.

La graine est fixe, mais le generateur xorshift64* differe de NumPy : les statistiques de recuperation de ce carnet sont celles affichees ci-dessus ; les valeurs de la source sont rappelees dans le paragraphe precedent.

## 3. La mesure NON triviale : recuperation stochastique (Langevin)

La recuperation *deterministe* (relaxation a convergence) est **degeneree** : a convergence, le franchissement du col est purement geometrique et la barriere n'a aucun effet — le test serait trivial et non falsifiable. La mesure honnete est **stochastique** : dynamique overdamped bruitee $dx = -V'(x)\,dt + \sqrt{2D\,dt}\,\xi$, ou les trois quantites ont un effet mesurable (escape d'Arrhenius $\sim e^{-\Delta V/D}$, taux de rappel $\sim e^{-\sigma t}$, geometrie). Sans bruit, pas de verdict falsifiable.

In [3]:
// Pourquoi le bruit est necessaire : deterministe (constant) vs stochastique (variance reelle).
var (aTemoin, bTemoin) = Bassins.Realiser(6.0, 1.0);
double xTemoin = Math.Sqrt(bTemoin / (2.0 * aTemoin));
double sigmaTemoin = 4.0 * bTemoin;
double largeurTemoin = xTemoin;
double barriereTemoin = bTemoin * bTemoin / (4.0 * aTemoin);
int retoursDet = 0;
for (int i = 0; i < 25; i++)
{
    double delta = 0.05 + (largeurTemoin * 1.2 - 0.05) * i / 24.0;
    double x = xTemoin - delta;
    for (int t = 0; t < 4000; t++)
        x += 0.01 * Bassins.Force(x, aTemoin, bTemoin);
    if (Math.Abs(x - xTemoin) < 0.05) retoursDet++;
}
double detTemoin = retoursDet / 25.0;
double stoTemoin = Mesures.Recuperation(xTemoin, aTemoin, bTemoin, new Alea(0), 0.35, 60, 3000);
Console.WriteLine($"substrat sigma={sigmaTemoin:F1}, width={largeurTemoin:F3}, barrier={barriereTemoin:F3}");
Console.WriteLine($"  recuperation deterministe = {detTemoin:F3}  (geometrique, insensible a la barriere)");
Console.WriteLine($"  recuperation stochastique = {stoTemoin:F3}  (depend de barriere, sigma ET largeur)");
Console.WriteLine($"  comparaison stochastique : source=0.333, ce carnet={stoTemoin:F3}, ecart={stoTemoin - 0.333:+0.000;-0.000;+0.000} (RNG different)");

substrat sigma=6.0, width=1.000, barrier=0.750


  recuperation deterministe = 0.800  (geometrique, insensible a la barriere)


  recuperation stochastique = 0.467  (depend de barriere, sigma ET largeur)


  comparaison stochastique : source=0.333, ce carnet=0.467, ecart=+0.134 (RNG different)


**Lecture de la sortie.** Sur le substrat temoin ($\sigma = 6{,}0$, $w = 1{,}000$, barrier $= 0{,}750$) : la recuperation **deterministe** vaut $0{,}800$, la recuperation **stochastique** $0{,}333$ -- un facteur $2{,}4$ d'ecart sur le meme substrat. La relaxation a convergence ne voit que la geometrie (le partage du domaine d'attraction), et la barriere ne change rien a ce partage : la mesure serait insensible a l'un des trois facteurs du probleme, donc non falsifiable sur lui. La dynamique bruitee, elle, franchit le col par tirages Arrhenius -- chaque facteur (barrier, $\sigma$, largeur) pese sur le taux -- et l'ecart mesure combien le bruit *revele* : la moitie de la recuperabilite apparente en deterministe etait un artefact de la convergence. C'est le choix de mesure qui decide si l'experience a un pouvoir discriminant.

Les valeurs stochastiques de ce paragraphe sont celles du jumeau Python ; la sortie precedente indique la valeur et l'ecart mesures ici avec le generateur xorshift64*.

> **Exercice 1** (cible barriere + largeur). Le decouplage par defaut fixe $(\sigma, w)$. On veut parfois cibler **(barriere $\Delta V$, largeur $w$)** directement — $\sigma$ devient une consequence. A partir de $\Delta V = b^2/(4a)$ et $w = \sqrt{b/(2a)}$, deroulez l'algebre pour obtenir $(a, b)$ puis la courbure resultante $\sigma$. Completez `realize_from_barrier_width` ci-dessous (les tests unitaires sont dans `tests/test_basin_family.py`).

In [4]:
// Exercice 1 -- a completer.
(double a, double b, double sigma) realize_from_barrier_width(double barrier, double width)
{
    // TODO etudiant : deroulez l'algebre.
    // Etape 1 : exprimer b en fonction de a et w.
    // Etape 2 : injecter dans barrier = b^2/(4a) et resoudre a.
    // Etape 3 : en deduire b puis sigma = 4b.
    return (0.0, 0.0, 0.0);
}
// Verification (a completer par l'etudiant) :
// var (a, b, s) = realize_from_barrier_width(2.0, 1.0);
// Console.WriteLine($"{a} {b} {s}"); // doit verifier barrier=b^2/(4a) et width=sqrt(b/(2a))
Console.WriteLine("Exercice a completer -- voir tests/test_basin_family.py");

Exercice a completer -- voir tests/test_basin_family.py


## 4. Le verdict decisif : correlation partielle a 2 covariables

Le Pont #1 controlait la largeur seule (1 covariable). Ici, la barriere co-varie avec le produit $\sigma \cdot w^2$ ; il faut la **purger**. On calcule donc la correlation partielle de Spearman de $\sigma$ avec la recuperation, en controlant **la largeur ET la barriere** (methode des residus de Frisch-Waugh-Lovell, generalisee a $N$ covariables). Le null est un test de permutation (brouillage de $\sigma$, 200 tirages, seuil p95).

Le verdict ternaire (jamais « promising ») :
- **CONFIRMED-NEGATIVE** : $\sigma$ n'a aucun pouvoir predictif propre -> le Pont #1 est un vrai negatif *general*.
- **SUBSTRATE-ARTIFACT** : $\sigma$ regagne un pouvoir propre -> le verdict fronce etait un artefact de couplage.
- **INCONCLUSIVE** : decouplage insuffisant ou signal inverse marque.

In [5]:
// Verdict principal, multi-seed (la recuperation stochastique est aleatoire :
// le seed traverse le calcul -> mesure de robustesse reelle, c.1014-L).
public static class StatistiquesPont
{
    // Projection sur une base orthonormee des rangs des covariables et de l'interception.
    public static double[] Residus(double[] y, params double[][] covariables)
    {
        int n = y.Length;
        var colonnes = new List<double[]> { Enumerable.Repeat(1.0, n).ToArray() };
        colonnes.AddRange(covariables.Select(Mesures.Rang));
        var baseOrtho = new List<double[]>();
        foreach (var colonne in colonnes)
        {
            var v = (double[])colonne.Clone();
            // Deux passes limitent la perte d'orthogonalite en cas de dependance.
            for (int passe = 0; passe < 2; passe++)
                foreach (var q in baseOrtho)
                {
                    double produit = 0.0;
                    for (int i = 0; i < n; i++) produit += v[i] * q[i];
                    for (int i = 0; i < n; i++) v[i] -= produit * q[i];
                }
            double norme = Math.Sqrt(v.Sum(x => x * x));
            if (norme > 1e-10)
                baseOrtho.Add(v.Select(x => x / norme).ToArray());
        }
        var resultat = (double[])y.Clone();
        foreach (var q in baseOrtho)
        {
            double produit = 0.0;
            for (int i = 0; i < n; i++) produit += resultat[i] * q[i];
            for (int i = 0; i < n; i++) resultat[i] -= produit * q[i];
        }
        return resultat;
    }

    public static double Partielle(double[] x, double[] y, params double[][] covariables)
    {
        if (x.Length < 2) return 0.0;
        double[] rx = Mesures.Rang(x), ry = Mesures.Rang(y);
        if (covariables.Length == 0) return Mesures.Pearson(rx, ry);
        return Mesures.Pearson(Residus(rx, covariables), Residus(ry, covariables));
    }

    public static double Spearman(double[] x, double[] y) => Mesures.Pearson(Mesures.Rang(x), Mesures.Rang(y));

    // Fisher-Yates avec rejet pour eviter un biais de reduction modulo.
    public static double[] Permutation(double[] valeurs, Alea rng)
    {
        var copie = (double[])valeurs.Clone();
        for (int i = copie.Length - 1; i > 0; i--)
        {
            ulong borne = (ulong)(i + 1);
            ulong seuil = unchecked(0UL - borne) % borne;
            ulong tirage;
            do { tirage = rng.Entier(); } while (tirage < seuil);
            int j = (int)(tirage % borne);
            double temp = copie[i]; copie[i] = copie[j]; copie[j] = temp;
        }
        return copie;
    }

    public static double Percentile95(double[] valeurs)
    {
        var tries = valeurs.OrderBy(x => x).ToArray();
        double position = 0.95 * (tries.Length - 1);
        int bas = (int)Math.Floor(position), haut = (int)Math.Ceiling(position);
        return tries[bas] + (tries[haut] - tries[bas]) * (position - bas);
    }

    public static Dictionary<string, double> Verdict(int seed, int nShuffle = 200, double noise = 0.35)
    {
        var famille = Mesures.Famille(new[] { 2.0, 3.0, 4.0, 6.0, 9.0, 13.0 }, new[] { 0.4, 0.6, 0.9, 1.3, 1.9 }, noise, 60, 3000, 0.02, 0.75, seed);
        double[] s = famille["sigma"], w = famille["width"], b = famille["barrier"], r = famille["recovery"];
        double p1 = Partielle(s, r, w), p2 = Partielle(s, r, w, b);
        double moyenne = r.Average();
        double ecartRecovery = Math.Sqrt(r.Select(x => (x - moyenne) * (x - moyenne)).Average());
        // Le generateur des permutations est initialise independamment, comme dans la source.
        var rng = new Alea(seed);
        double[] nulls = new double[nShuffle];
        for (int i = 0; i < nShuffle; i++) nulls[i] = Math.Abs(Partielle(Permutation(s, rng), r, w, b));
        return new Dictionary<string, double> {
            ["sw"] = Spearman(s, w), ["sr"] = Spearman(s, r), ["wr"] = Spearman(w, r),
            ["br"] = Spearman(b, r), ["p1"] = p1, ["p2"] = p2, ["p95"] = Percentile95(nulls),
            ["recoveryStd"] = ecartRecovery
        };
    }

    public static string Libelle(Dictionary<string, double> v)
    {
        // Un decouplage insuffisant ou une recuperation constante ne permet pas de conclure.
        if (Math.Abs(v["sw"]) >= 0.2 || v["recoveryStd"] < 1e-12) return "INCONCLUSIVE";
        if (v["p2"] > v["p95"] && v["p2"] > 0.2) return "SUBSTRATE-ARTIFACT";
        if (v["p2"] < -0.2) return "INCONCLUSIVE";
        return "CONFIRMED-NEGATIVE";
    }
}

var rows = new List<Dictionary<string, double>>();
var graines = new[] { 0, 1, 7, 42 };
// References Python arrondies : les differences sont calculees sur les valeurs C#.
double[] referenceP2 = { 0.173, 0.170, 0.128, 0.136 };
double[] referenceP95 = { 0.324, 0.385, 0.360, 0.372 };
for (int i = 0; i < graines.Length; i++)
{
    int seed = graines[i];
    var v = StatistiquesPont.Verdict(seed, 200);
    rows.Add(v);
    Console.WriteLine($"seed={seed,2} | decouplage rho(sw)={v["sw"]:+0.000;-0.000;+0.000} | brut rho(sr)={v["sr"]:+0.000;-0.000;+0.000} rho(wr)={v["wr"]:+0.000;-0.000;+0.000} rho(br)={v["br"]:+0.000;-0.000;+0.000} | partial 1-cov(w)={v["p1"]:+0.000;-0.000;+0.000}  2-cov(w,b)={v["p2"]:+0.000;-0.000;+0.000} [null p95={v["p95"]:F3}] -> {StatistiquesPont.Libelle(v)}");
    Console.WriteLine($"  ecarts au Python arrondi : partielle={v["p2"] - referenceP2[i]:+0.000;-0.000;+0.000}, p95={v["p95"] - referenceP95[i]:+0.000;-0.000;+0.000} (RNG different)");
}
var verdicts = rows.Select(StatistiquesPont.Libelle).ToArray();
var dominant = verdicts.GroupBy(x => x).OrderByDescending(g => g.Count()).First().Key;
Console.WriteLine($"\nVerdict dominant : {dominant} ({verdicts.Count(x => x == "CONFIRMED-NEGATIVE")}/{verdicts.Length} seeds CONFIRMED-NEGATIVE)");

seed= 0 | decouplage rho(sw)=+0.000 | brut rho(sr)=+0.091 rho(wr)=+0.661 rho(br)=+0.613 | partial 1-cov(w)=+0.122  2-cov(w,b)=+0.144 [null p95=0.408] -> CONFIRMED-NEGATIVE


  ecarts au Python arrondi : partielle=-0.029, p95=+0.084 (RNG different)


seed= 1 | decouplage rho(sw)=+0.000 | brut rho(sr)=+0.256 rho(wr)=+0.627 rho(br)=+0.689 | partial 1-cov(w)=+0.328  2-cov(w,b)=-0.135 [null p95=0.375] -> CONFIRMED-NEGATIVE


  ecarts au Python arrondi : partielle=-0.305, p95=-0.010 (RNG different)


seed= 7 | decouplage rho(sw)=+0.000 | brut rho(sr)=+0.275 rho(wr)=+0.683 rho(br)=+0.736 | partial 1-cov(w)=+0.376  2-cov(w,b)=+0.001 [null p95=0.370] -> CONFIRMED-NEGATIVE


  ecarts au Python arrondi : partielle=-0.127, p95=+0.010 (RNG different)


seed=42 | decouplage rho(sw)=+0.000 | brut rho(sr)=+0.292 rho(wr)=+0.519 rho(br)=+0.577 | partial 1-cov(w)=+0.342  2-cov(w,b)=+0.229 [null p95=0.384] -> CONFIRMED-NEGATIVE


  ecarts au Python arrondi : partielle=+0.093, p95=+0.012 (RNG different)



Verdict dominant : CONFIRMED-NEGATIVE (4/4 seeds CONFIRMED-NEGATIVE)


### Interpretation

Sur les 4 graines, la **correlation partielle a 2 covariables** `partial_rho_given_width_barrier` reste proche de zero et sous le null p95 : meme apres avoir decouple $\sigma$ de la largeur *et* purge la barriere co-variante, $\sigma$ n'apporte aucun pouvoir predictif propre sur la recuperabilite. Le verdict est **CONFIRMED-NEGATIVE**.

Conclusion : la falsification du Pont #1 sur la fronce n'etait **pas** un artefact de couplage. C'est un **vrai negatif general** — la courbure locale $\sigma$ est un *proxy correle* de la geometrie du bassin (largeur/barriere), non la cause de la recuperabilite. C'est le point methodologique de [#8077](https://github.com/jsboige/CoursIA/issues/8077) : un pont falsifie est aussi informatif qu'un pont confirme.

> Note honnete : la 1-covariable (largeur seule) laisse fuir de la barriere et donne une partielle legerement positive — c'est precisement le piege que la 2-covariables resout. Comparer les deux colonnes dans le tableau ci-dessus.

Cette interpretation rapporte les resultats du jumeau Python. Les sorties ci-dessus affichent les resultats et les ecarts mesures avec le generateur C# ; ils peuvent differer pour les quantites stochastiques.

> **Exercice 2** (domaine de bruit). Le verdict depend-il du regime de bruit ? Pour `noise` dans `[0.15, 0.3, 0.5, 0.7]`, executez `StatistiquesPont.Verdict` et tracez `p2` contre `noise`. Documentez le domaine ou le verdict reste CONFIRMED-NEGATIVE et celui ou il devient INCONCLUSIVE (bruit trop faible = recuperation saturee a 1, variance nulle).

In [6]:
// Exercice 2 -- a completer.
List<(double noise, double partial, string verdict)> phase_diagram_noise(double[] noises)
{
    var results = new List<(double noise, double partial, string verdict)>();
    // TODO etudiant : boucler sur noises, appeler StatistiquesPont.Verdict, collecter.
    return results;
}
// var noises = new[] { 0.15, 0.3, 0.5, 0.7 };
// var res = phase_diagram_noise(noises);
// foreach (var (noise, partial, verdict) in res)
//     Console.WriteLine($"noise={noise}: partial={partial:+0.000;-0.000;+0.000} -> {verdict}");
Console.WriteLine("Exercice a completer");

Exercice a completer


## 5. Controle nul : la sous-famille re-couplee reproduit la fronce

Discipline du null model (#9531) : si le protocole est sain, une sous-famille ou $\sigma$ et $w$ **re-derivent ensemble** ($a$ fixe, $b$ varie) doit reproduire le motif fronce — couplage eleve ET partielle a 1 covariable $\approx 0$. Si ce controle echouait, le decouplage de la famille principale serait suspect.

In [7]:
// Controle nul : a fixe, b varie ; une seule trajectoire de tirages pour la famille.
var rngNull = new Alea(0);
var sigmasNull = new List<double>();
var widthsNull = new List<double>();
var recNull = new List<double>();
foreach (double bNull in new[] { 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5 })
{
    double aNull = 1.0;
    double xNull = Math.Sqrt(bNull / (2.0 * aNull));
    sigmasNull.Add(4.0 * bNull);
    widthsNull.Add(xNull);
    recNull.Add(Mesures.Recuperation(xNull, aNull, bNull, rngNull, 0.35, 60, 3000, 0.02, 0.75));
}
double[] sigNull = sigmasNull.ToArray(), widNull = widthsNull.ToArray(), recoveryNull = recNull.ToArray();
double rhoNullSW = StatistiquesPont.Spearman(sigNull, widNull);
double rhoNullWR = StatistiquesPont.Spearman(widNull, recoveryNull);
double partialNull = StatistiquesPont.Partielle(sigNull, recoveryNull, widNull);
// Le null utilise le meme generateur apres les trajectoires, comme la source.
var permutationsNull = new double[200];
for (int i = 0; i < permutationsNull.Length; i++)
    permutationsNull[i] = Math.Abs(StatistiquesPont.Partielle(StatistiquesPont.Permutation(sigNull, rngNull), recoveryNull, widNull));
double p95Null = StatistiquesPont.Percentile95(permutationsNull);
bool motifFronce = rhoNullSW > 0.6 && Math.Abs(partialNull) < 0.3;
Console.WriteLine("Controle nul re-couple (a fixe, b varie) :");
Console.WriteLine($"  rho(sigma, width)        = {rhoNullSW:+0.000;-0.000;+0.000}  (re-couple : > 0.6)");
Console.WriteLine($"  rho(width, recovery)    = {rhoNullWR:+0.000;-0.000;+0.000}");
Console.WriteLine($"  partial(sigma | width)  = {partialNull:+0.000;-0.000;+0.000}  (fronce : |.| < 0.3)");
Console.WriteLine($"  reproduit motif fronce  = {motifFronce}");
Console.WriteLine($"  ecart rho(width, recovery) au Python arrondi (+0.714) = {rhoNullWR - 0.714:+0.000;-0.000;+0.000} (RNG different)");
// Le contraste utilise les memes parametres que le verdict principal.
var vMain = StatistiquesPont.Verdict(0, 200);
Console.WriteLine($"\nContraste famille decouplee : rho(sigma,width) = {vMain["sw"]:+0.000;-0.000;+0.000} (decouple)");

Controle nul re-couple (a fixe, b varie) :


  rho(sigma, width)        = +1.000  (re-couple : > 0.6)


  rho(width, recovery)    = +0.893


  partial(sigma | width)  = +0.000  (fronce : |.| < 0.3)


  reproduit motif fronce  = True


  ecart rho(width, recovery) au Python arrondi (+0.714) = +0.179 (RNG different)



Contraste famille decouplee : rho(sigma,width) = +0.000 (decouple)


**Lecture du controle nul, chiffre par chiffre.** Dans la sous-famille re-couplee ($a$ fixe, $b$ varie) : $\rho(\sigma, w) = +1{,}000$ -- le re-couplage est total (la fronce originelle liait courbure et largeur, on la reconstruit a dessein) ; $\rho(w, r) = +0{,}714$ -- la largeur seule explique les deux tiers de la variance de recuperation ; et $\rho(\sigma \mid w) = +0{,}000$ : une fois la largeur purgee, la courbure n'apporte *rien*, et le detecteur de motif confirme la reproduction de la fronce (`True`). Mis en regard de la ligne de contraste ci-dessus (famille decouplee : $\rho(\sigma, w) = +0{,}000$), le tableau est complet : en regime *couple*, le pouvoir propre de $\sigma$ est exactement nul ($+0{,}000$) ; en regime *decouple*, la correlation partielle a 2 covariables de la section 4 reste sous le null p95 sur les 4 graines ($0{,}128$ a $0{,}173$ contre $0{,}324$ a $0{,}385$) -- deux regimes de couplage opposes, meme absence de pouvoir propre. La falsification du Pont #1 ne depend pas du couplage : c'est bien un negatif *general*.

Ces chiffres sont ceux du jumeau Python. Le generateur C# est different : la sortie precedente affiche la correlation stochastique mesuree ici et son ecart a la reference.

> **Exercice 3** (suffisance du controle a 2 covariables). Le verdict controle (largeur, barriere). Une 3e quantite geometrique — la **profondeur du puits** $|V(x^*)|$ normalisee — pourrait-elle restaurer un pouvoir propre a $\sigma$ ? Ajoutez-la comme 3e covariable dans `StatistiquesPont.Partielle` (qui accepte un nombre quelconque de covariables via FWL) et executez. Que conclure sur la suffisance du controle a 2 covariables ?

In [8]:
// Exercice 3 -- a completer.
// TODO etudiant : examiner la profondeur du puits.
// Question : profondeur et barriere sont-elles ici la MEME quantite ? Si oui,
// ajouter la profondeur comme 3e covariable change-t-elle quelque chose ?
// Ecrire un court raisonnement (en commentaire) puis une verification numerique.
Console.WriteLine("Exercice a completer -- reflexion sur la redondance des covariables");

Exercice a completer -- reflexion sur la redondance des covariables


## Vérifiez votre compréhension

1. Le découplage constructif donne `corr(sigma, width) = +0.000` mais `corr(sigma, barrier) = +0.463`. Pourquoi la fuite de barrière force-t-elle une corrélation partielle à 2 covariables (FWL) plutôt qu'un simple contrôle de la largeur ?
2. La récupération déterministe du témoin vaut `0.800` contre `0.467` en stochastique ; sur la famille, l'écart-type de la récupération vaut `0.186`. La récupération stochastique du témoin vaut `0.333` dans le jumeau Python : les générateurs diffèrent malgré la graine fixe. Qu'est-ce qui rend la mesure stochastique *non triviale* là où la déterministe est dégénérée ?
3. Le contrôle nul re-couplé rend `rho(sigma, width) = +1.000` et `partial = +0.000`. Si le verdict dominant est CONFIRMED-NEGATIVE (4/4 seeds), que reproduit exactement le contrôle — et pourquoi est-ce lui qui valide le protocole ?

## À retenir

Le Pont #1-bis tranche : sur une famille de substrats ou la courbure $\sigma$ et la largeur de bassin varient *independamment par construction*, et apres purge de la barriere co-variante, **$\sigma$ n'a aucun pouvoir predictif propre sur la recuperabilite**. La falsification du Pont #1 acquise sur la fronce est donc un **vrai negatif general**, non un artefact de couplage.

Leçon methodologique :
1. **Un substrat unique ne tranche pas la generalite** : la fronce couplait $\sigma$ et largeur ; il fallait une famille decouplee pour decider.
2. **La rigueur statistique doit suivre la geometrie** : la barriere co-varie avec le produit des cadrans, donc controler la largeur seul laisse fuir un confondant. La correlation partielle a 2 covariables (FWL) est l'outil decisif.
3. **Une mesure doit etre non triviale** : la recuperation deterministe est degeneree (geometrique) ; le verdict falsifiable exige la dynamique stochastique.

Suite : generaliser a d'autres formes de substrats (puits asymetriques, paysages 2D) pour eprouver encore la generalite du negatif.

### Parité avec le jumeau Python

Les écarts stochastiques sont calculés et affichés dans les sections correspondantes : récupération du témoin, corrélations partielles et seuils multi-graines, puis contrôle re-couplé. Les générateurs étant différents, l'équivalence porte sur le protocole et non sur l'identité des tirages.

## Pour aller plus loin

- Le régime asymétrique (bassins non interchangeables) : [ICT-15h](ICT-15h-Bridge1bis-AsymmetricFamily-Python.ipynb).
- Le paysage 2D anisotrope (knob transverse) : [ICT-15i](ICT-15i-Bridge1bis-2DLandscape-Python.ipynb).
- **Jumeau Python** → [ICT-15f-Bridge1bis-DecoupledFamily-Python](ICT-15f-Bridge1bis-DecoupledFamily-Python.ipynb) — le carnet d'origine, en Python.
- Le module `ict.basin_family` (référencé en tête de carnet) porte la famille de bassins.
- Revenir au [cadrage de la série](../ICT-0-Framing.md).